# Hotel Booking Cancellation Prediction & Decision System

## 1. Business Problem

Hotel booking cancellations can lead to unused rooms, revenue loss, and inefficient inventory planning. Predicting whether a booking is likely to be cancelled can help hotels take preventive actions such as confirmation follow-ups, deposit requirements, or targeted retention strategies.

The objective of this project is to develop a machine learning system that predicts the probability of a hotel booking being cancelled using information available at the time of booking.

The project follows an end-to-end machine learning workflow:

* Data cleaning and preprocessing
* Exploratory data analysis
* Feature engineering
* Classification model development
* Model evaluation
* Cancellation probability prediction
* Risk-based business recommendations

The final system will classify bookings into cancellation-risk levels and provide an actionable recommendation based on the predicted probability.



In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'pandas'

In [ ]:
import os
import sys

print(sys.executable)
print(os.getcwd())

C:\Users\KARTIK SRIBVASTAVA\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
c:\Users\KARTIK SRIBVASTAVA\OneDrive\Desktop\studeies\ml\practise project\hotel booking anal


In [ ]:
from google.colab import files
files.upload()

In [ ]:
data = pd.read_csv("hotel_bookings.csv")

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.isnull().sum().sort_values(ascending = False)

In [ ]:
data['company'].value_counts(dropna = False)

In [ ]:
data["company"].isna().value_counts()

In [ ]:
pd.crosstab(data["customer_type"],data["company"].isna(),normalize="index")*100

In [ ]:
data['agent'].value_counts(dropna = False)

In [ ]:
data["agent"].isna().value_counts()

In [ ]:
pd.crosstab(data['distribution_channel'],data['agent'].isna(),normalize=True)*100

In [ ]:
pd.crosstab(
    data["market_segment"],
    data["agent"].isna(),
    normalize="index"
) * 100

In [ ]:
data['has_agent'] = data['agent'].notna().astype(int)

In [ ]:
data["country"].value_counts(dropna=False).head(20)

In [ ]:
data["country"].isna().value_counts()

In [ ]:
data['country'] = data['country'].fillna('unknown')

In [ ]:
data['country'].isna().value_counts()

In [ ]:
data["children"].value_counts(dropna=False)

In [ ]:
data[data["children"].isna()]

In [ ]:
data = data.dropna(subset = ['children'])

In [ ]:
data.duplicated().value_counts()

In [ ]:
data.duplicated().value_counts()

In [ ]:
data.shape

In [ ]:
data.duplicated().sum()

In [ ]:
data['reservation_status'].unique()

In [ ]:
data["reservation_status"].value_counts()

In [ ]:
data.groupby("reservation_status")["is_canceled"].value_counts(normalize = True)

In [ ]:
leakage_cols = [
    "reservation_status",
    "reservation_status_date"
]

In [ ]:
data["is_canceled"].value_counts(normalize=True) * 100

In [ ]:
data.info()

In [ ]:
pd.crosstab(data['hotel'],data['is_canceled'],normalize = 'index')*100

In [ ]:


cancellation_by_hotel = (
    data.groupby("hotel")["is_canceled"]
    .mean()
    .sort_values(ascending=False)
)

sns.barplot(data = cancellation_by_hotel.reset_index(), x = "hotel", y = "is_canceled")

plt.ylabel("Cancellation Rate")
plt.xlabel("Hotel")
plt.title("Cancellation Rate by Hotel")
plt.xticks(rotation=0)
plt.show()

In [ ]:
data.groupby("is_canceled")["lead_time"].describe()

In [ ]:
bins = [0, 7, 30, 90, 180, float("inf")]
labels = [
    "0-7 days",
    "8-30 days",
    "31-90 days",
    "91-180 days",
    "181+ days"
]

data["lead_time_group"] = pd.cut(
    data["lead_time"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
lead_time_cancellation = (
    data.groupby("lead_time_group")["is_canceled"]
    .mean() * 100
)

lead_time_cancellation

In [ ]:
pd.crosstab(data['deposit_type'],data['is_canceled'],normalize = 'index')*100

In [ ]:
data.groupby('deposit_type')['is_canceled'].agg(['count','sum'])

In [ ]:
pd.crosstab(
    data["customer_type"],
    data["is_canceled"],
    normalize="index"
) * 100

In [ ]:
pd.crosstab(
    data["market_segment"],
    data["is_canceled"],
    normalize="index"
) * 100

In [ ]:
data["market_segment"].value_counts()

In [ ]:
data.groupby("previous_cancellations")["is_canceled"].agg(
    ["count", "sum","mean"]
).sort_index()

In [ ]:
pd.crosstab(
    data["is_repeated_guest"],
    data["is_canceled"],
)

In [ ]:
result = data.groupby("total_of_special_requests")["is_canceled"].agg(
    count="count",
    canceled="sum",
    cancellation_rate="mean"
)
result['cancellation_rate'] *=100
result

In [ ]:
result = data.groupby("previous_cancellations")["is_canceled"].agg(
    total="count",
    canceled="sum"
)

result["not_canceled"] = result["total"] - result["canceled"]
result["cancellation_rate"] = result["canceled"] / result["total"] * 100

result

In [ ]:

data = data.drop(columns=leakage_cols)

data.shape

In [ ]:
data.dtypes

In [ ]:
data.describe().T

In [ ]:
# Feature engineering

data["total_nights"] = (
    data["stays_in_weekend_nights"] +
    data["stays_in_week_nights"]
)

data["total_guests"] = (
    data["adults"] +
    data["children"].fillna(0) +
    data["babies"]
)

data["total_previous_bookings"] = (
    data["previous_cancellations"] +
    data["previous_bookings_not_canceled"]
)

data["has_previous_booking"] = (
    data["total_previous_bookings"] > 0
).astype(int)

data["has_special_requests"] = (
    data["total_of_special_requests"] > 0
).astype(int)

data["is_room_changed"] = (
    data["reserved_room_type"] != data["assigned_room_type"]
).astype(int)

data.head()

In [ ]:
target = "is_canceled"

In [ ]:
drop_cols = [
    "is_canceled",
    "lead_time_group",
    "agent",
    "company"
]

X = data.drop(columns=drop_cols)
y = data["is_canceled"]

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
X.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        ))
    ]
)
from xgboost import XGBClassifier

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss"
        ))
    ]
)


In [ ]:
logistic_pipeline.fit(X_train, y_train)
xgb_pipeline.fit(X_train, y_train)
random_forest_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }
results = {}

results["Logistic Regression"] = evaluate_model(
    logistic_pipeline, X_test, y_test
)

results["Random Forest"] = evaluate_model(
    random_forest_pipeline, X_test, y_test
)
results["XGBoost"] = evaluate_model(
    xgb_pipeline,
    X_test,
    y_test
)

results_df = pd.DataFrame(results).T
results_df

In [ ]:
rf_model = random_forest_pipeline.named_steps["model"]

feature_names = (
    random_forest_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(
    by="Importance",
    ascending=False
)

feature_importance_df.head(20)

In [ ]:
def cancellation_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.40:
        return "Medium Risk"
    else:
        return "Low Risk"


def booking_recommendation(probability):
    if probability >= 0.70:
        return "Require confirmation/deposit or proactive follow-up"
    elif probability >= 0.40:
        return "Send confirmation reminder"
    else:
        return "Normal booking handling"

In [ ]:
def predict_booking_risk(booking):
    probability = random_forest_pipeline.predict_proba(booking)[0, 1]

    risk = cancellation_risk(probability)
    recommendation = booking_recommendation(probability)

    return {
        "Cancellation Probability": round(probability, 3),
        "Risk Level": risk,
        "Recommendation": recommendation
    }

In [ ]:
sample_booking = X_test.iloc[[0]]

prediction = predict_booking_risk(sample_booking)

prediction

In [ ]:
import joblib

joblib.dump(
    random_forest_pipeline,
    "hotel_cancellation_model.pkl"
)

In [ ]:
loaded_model = joblib.load(
    "hotel_cancellation_model.pkl"
)

In [ ]:
loaded_model = joblib.load(
    "hotel_cancellation_model.pkl"
)